# Linear Regression — From MLE to Gradient Descent

This notebook accompanies the blog post on Linear Regression. We derive the MSE loss from Maximum Likelihood Estimation under Gaussian noise, then fit a polynomial regression model with PyTorch using gradient descent.

### Setup

We generate synthetic data from a 7th-degree polynomial:

$$
y = \sum_{d=0}^{6} \theta_d x^d + \epsilon, \qquad \epsilon \sim \mathcal{N}(0, \sigma^2)
$$

The goal is to recover the coefficients $\theta$ by minimizing the MSE loss, which is equivalent to maximizing the Gaussian log-likelihood.

### MLE derivation (recap)

Assuming $y \mid x; \theta \sim \mathcal{N}(h_\theta(x), \sigma^2)$, the log-likelihood for the dataset is:

$$
\ell(\theta) = -\frac{N}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_{i=1}^{N}(y_i - h_\theta(x_i))^2
$$

Maximizing $\ell(\theta)$ w.r.t. $\theta$ is the same as minimizing:

$$
\text{MSE}(\theta) = \frac{1}{N}\sum_{i=1}^{N}(y_i - h_\theta(x_i))^2
$$

This gives us a principled justification for using MSE in linear regression.

In [8]:
import torch as t
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from torch import nn
from torch.nn import Linear
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

SEED = 42
t.manual_seed(SEED)
np.random.seed(SEED)

# Site color palette (matches the blog theme)
SITE = dict(
    bg_primary   = '#213636',
    bg_secondary = '#1a2d2d',
    bg_tertiary  = '#2a4444',
    border       = '#3a5454',
    text_primary = '#c8c4b8',
    text_secondary = '#a09890',
    accent       = '#bdb76b',
    olive        = '#828631',
)

PLOTS_DIR = Path("./plots")
PLOTS_DIR.mkdir(exist_ok=True)

### Dataset

We build a `PolynomialDataset` that samples $x$ uniformly in $[-1, 1]$, constructs polynomial features $[1, x, x^2, \dots, x^{D-1}]$, and computes targets with Gaussian noise.

In [9]:
class PolynomialDataset(Dataset):
    def __init__(self, degree: int, data_size=1000, coefs=None):
        super().__init__()
        if coefs is None:
            coefs = t.randn(degree) * 10
        else:
            coefs = t.tensor(coefs, dtype=t.float32)

        self.coefs = coefs
        self.eps = t.randn(2 * data_size + 1) * 0.1
        x = t.linspace(-1, 1, steps=2 * data_size + 1)
        self.x = t.stack([x ** idx for idx in range(self.coefs.size(0))], dim=1)
        self.y = self.x @ self.coefs + self.eps

    def __len__(self):
        return len(self.x)

    def __getitem__(self, index):
        return self.x[index], self.y[index]


DEGREE = 5
dataset = PolynomialDataset(degree=DEGREE)
print(f"Dataset size: {len(dataset)}")
print(f"True coefficients: {dataset.coefs.numpy()}")

Dataset size: 2001
True coefficients: [  3.3669038   1.288094    2.3446236   2.3033304 -11.228563 ]


### Visualize the dataset

Plot a subset of the noisy $(x, y)$ samples along with the true polynomial curve.

In [10]:
def plot_dataset(dataset, max_points=2000):
    x = dataset.x[:, 1].numpy()  # original x feature
    y = dataset.y.numpy()
    y_true = (dataset.x @ dataset.coefs).numpy()

    idx = np.random.choice(len(x), size=min(max_points, len(x)), replace=False)
    idx.sort()

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x[idx], y=y[idx],
        mode='markers',
        marker=dict(color=SITE['accent'], size=4, opacity=0.6),
        name='Noisy samples'
    ))
    fig.add_trace(go.Scatter(
        x=x, y=y_true,
        mode='lines',
        line=dict(color=SITE['olive'], width=2),
        name='True polynomial'
    ))
    fig.update_layout(
        title=dict(text='Synthetic Polynomial Dataset', font=dict(color=SITE['text_primary'])),
        xaxis=dict(title='x', color=SITE['text_secondary'], gridcolor=SITE['border']),
        yaxis=dict(title='y', color=SITE['text_secondary'], gridcolor=SITE['border']),
        paper_bgcolor=SITE['bg_secondary'],
        plot_bgcolor=SITE['bg_primary'],
        font=dict(color=SITE['text_primary']),
        legend=dict(x=0.02, y=0.98, bgcolor=SITE['bg_secondary'], bordercolor=SITE['border']),
        autosize=True,
        margin=dict(l=60, r=20, t=60, b=40),
    )
    return fig

fig_dataset = plot_dataset(dataset)
fig_dataset.write_html(
    PLOTS_DIR / "linear_regression_dataset.html",
    config=dict(responsive=True, displayModeBar=True),
    include_plotlyjs='cdn',
)
fig_dataset.show()

### Model

A simple linear regression model with no bias term. The bias is absorbed into the feature vector (the $x^0 = 1$ column).

In [11]:
class LinearRegression(nn.Module):
    def __init__(self, in_feats, out_feats=1):
        super().__init__()
        self.linear = Linear(in_feats, out_feats, bias=False)

    def forward(self, x):
        return self.linear(x).squeeze()


model = LinearRegression(in_feats=DEGREE)
print(model)

LinearRegression(
  (linear): Linear(in_features=5, out_features=1, bias=False)
)


### Trainer

We train with the MSE loss (derived from MLE) and the AdamW optimizer. We track training loss, validation loss, and the $R^2$ score on the validation set.

In [12]:
class Trainer:
    def __init__(self, model, train_dataset, val_dataset, epochs=100, lr=1e-3, eps=1e-7, batch_size=32):
        self.model = model
        self.train_dataset = train_dataset
        self.val_dataset = val_dataset
        self.epochs = epochs
        self.batch_size = batch_size
        self.optim = AdamW(params=model.parameters(), lr=lr, eps=eps)

    def mse_loss(self, target, predicted):
        return t.pow(target - predicted, 2).mean()

    def r2_score(self, target, predicted):
        ss_res = t.pow(target - predicted, 2).sum()
        ss_tot = t.pow(target - target.mean(), 2).sum()
        return (1 - ss_res / ss_tot).item()

    def plot_loss_curves(self, train_losses, val_losses, val_r2_scores):
        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=('MSE Loss', r'$R^2$ Score'),
            horizontal_spacing=0.12,
        )
        epochs = list(range(1, len(train_losses) + 1))
        fig.add_trace(go.Scatter(x=epochs, y=train_losses, mode='lines', name='Train loss',
                                 line=dict(color=SITE['accent'])), row=1, col=1)
        fig.add_trace(go.Scatter(x=epochs, y=val_losses, mode='lines', name='Val loss',
                                 line=dict(color=SITE['olive'])), row=1, col=1)
        fig.add_trace(go.Scatter(x=epochs, y=val_r2_scores, mode='lines', name='Val R²',
                                 line=dict(color=SITE['accent'])), row=1, col=2)

        fig.update_layout(
            title=dict(text='Training Progress', font=dict(color=SITE['text_primary'])),
            paper_bgcolor=SITE['bg_secondary'],
            plot_bgcolor=SITE['bg_primary'],
            font=dict(color=SITE['text_primary']),
            legend=dict(orientation='h', x=0.5, y=-0.15, xanchor='center', yanchor='top',
                        bgcolor=SITE['bg_secondary'], bordercolor=SITE['border']),
            autosize=True,
            margin=dict(l=60, r=40, t=80, b=60),
        )
        for axis in ['xaxis', 'yaxis', 'xaxis2', 'yaxis2']:
            fig.update_layout({axis: dict(gridcolor=SITE['border'], color=SITE['text_secondary'])})
        return fig

    def plot_model_fit(self, dataset):
        self.model.eval()
        with t.no_grad():
            x = dataset.x[:, 1].numpy()
            y_true = dataset.y.numpy()
            y_pred = self.model(dataset.x).numpy()

        fig = go.Figure()
        fig.add_trace(go.Scatter(x=x, y=y_true, mode='markers', name='True noisy y',
                                 marker=dict(color=SITE['olive'], size=4, opacity=0.5)))
        fig.add_trace(go.Scatter(x=x[np.argsort(x)], y=y_pred[np.argsort(x)], mode='lines', name='Model fit',
                                 line=dict(color=SITE['accent'], width=2)))
        fig.update_layout(
            title=dict(text='Model Fit on Full Dataset', font=dict(color=SITE['text_primary'])),
            xaxis=dict(title='x', color=SITE['text_secondary'], gridcolor=SITE['border']),
            yaxis=dict(title='y', color=SITE['text_secondary'], gridcolor=SITE['border']),
            paper_bgcolor=SITE['bg_secondary'],
            plot_bgcolor=SITE['bg_primary'],
            font=dict(color=SITE['text_primary']),
            legend=dict(orientation='h', x=0.5, y=-0.15, xanchor='center', yanchor='top',
                        bgcolor=SITE['bg_secondary'], bordercolor=SITE['border']),
            autosize=True,
            margin=dict(l=60, r=20, t=60, b=60),
        )
        return fig

    def train(self):
        train_loader = DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)
        val_loader = DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False)

        train_losses, val_losses, val_r2_scores = [], [], []

        for epoch in tqdm(range(self.epochs), desc='Epochs'):
            self.model.train()
            batch_losses = []
            for X_train, y_train in train_loader:
                predicted = self.model(X_train)
                loss = self.mse_loss(y_train, predicted)
                self.optim.zero_grad()
                loss.backward()
                self.optim.step()
                batch_losses.append(loss.item())
            train_losses.append(sum(batch_losses) / len(batch_losses))

            self.model.eval()
            with t.no_grad():
                all_preds, all_targets = [], []
                for X_val, y_val in val_loader:
                    preds = self.model(X_val)
                    all_preds.append(preds)
                    all_targets.append(y_val)
                all_preds = t.cat(all_preds)
                all_targets = t.cat(all_targets)
                val_losses.append(self.mse_loss(all_targets, all_preds).item())
                val_r2_scores.append(self.r2_score(all_targets, all_preds))

            if (epoch + 1) % 100 == 0:
                print(f"Epoch {epoch+1}/{self.epochs} | train_loss: {train_losses[-1]:.4f} | "
                      f"val_loss: {val_losses[-1]:.4f} | val_R²: {val_r2_scores[-1]:.4f}")

        return self.model, self.plot_loss_curves(train_losses, val_losses, val_r2_scores), self.plot_model_fit(dataset)

### Train the model

Split the dataset 80/20 and run gradient descent for 10,000 epochs.

In [13]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

model = LinearRegression(in_feats=DEGREE)
trainer = Trainer(model, train_ds, val_ds, epochs=600, lr=1e-3)
model, loss_curves, model_fit = trainer.train()

loss_curves.write_html(
    PLOTS_DIR / "linear_regression_loss_curves.html",
    config=dict(responsive=True, displayModeBar=True),
    include_plotlyjs='cdn',
)
model_fit.write_html(
    PLOTS_DIR / "linear_regression_model_fit.html",
    config=dict(responsive=True, displayModeBar=True),
    include_plotlyjs='cdn',
)

loss_curves.show()
model_fit.show()

print("\nLearned coefficients:")
print(model.linear.weight.detach().numpy().squeeze())
print("\nTrue coefficients:")
print(dataset.coefs.numpy())

Epochs:  20%|█▉        | 117/600 [00:00<00:03, 158.25it/s]

Epoch 100/600 | train_loss: 0.7722 | val_loss: 0.8351 | val_R²: 0.9118


Epochs:  38%|███▊      | 231/600 [00:01<00:02, 159.25it/s]

Epoch 200/600 | train_loss: 0.2173 | val_loss: 0.2133 | val_R²: 0.9775


Epochs:  55%|█████▌    | 331/600 [00:02<00:01, 159.68it/s]

...

Epochs:  88%|████████▊ | 530/600 [00:03<00:00, 158.63it/s]

Epoch 500/600 | train_loss: 0.0173 | val_loss: 0.0170 | val_R²: 0.9982


Epochs: 100%|██████████| 600/600 [00:03<00:00, 158.02it/s]

Epoch 600/600 | train_loss: 0.0105 | val_loss: 0.0111 | val_R²: 0.9988



Learned coefficients:
[  3.397691    1.2856877   2.0629303   2.292855  -10.917453 ]

True coefficients:
[  3.3669038   1.288094    2.3446236   2.3033304 -11.228563 ]


### Closed-form solution (Normal Equations)

For comparison, we can also solve linear regression in closed form:

$$
\hat{\theta} = (X^T X)^{-1} X^T y
$$

This is useful as a sanity check for the gradient-descent solution.

In [14]:
X_all = dataset.x.numpy()
y_all = dataset.y.numpy()

theta_closed = np.linalg.pinv(X_all.T @ X_all) @ X_all.T @ y_all

print("Closed-form coefficients:")
print(theta_closed)
print("\nGradient-descent coefficients:")
print(model.linear.weight.detach().numpy().squeeze())
print("\nTrue coefficients:")
print(dataset.coefs.numpy())

Closed-form coefficients:
[  3.3656242   1.2726088   2.3478703   2.312323  -11.234924 ]

Gradient-descent coefficients:
[  3.397691    1.2856877   2.0629303   2.292855  -10.917453 ]

True coefficients:
[  3.3669038   1.288094    2.3446236   2.3033304 -11.228563 ]
